# Raw-сверка Voice ↔ CRM по Id задачи и Id ПрПр

Минимальный notebook для Omega.

Цель: **без LLM и без аналитических преобразований** взять исходные CRM-файлы `part-*.csv`,
найти в них строки, которые напрямую совпадают с Voice по:

- `Voice.Id задачи` ↔ `CRM taskid` / `CRM key`;
- `Voice.Id ПрПр` ↔ `CRM productOfferId` / `CRM key`;

и сохранить эти CRM-строки в Excel для ручной проверки глазами.

На выходе создаётся папка `omega_raw_match_export/`:

- `01_voice_match_keys.xlsx` — какие ключи взяли из Voice;
- `02_crm_task_rows_by_voice_id_task.xlsx` — исходные CRM-строки задач, совпавшие с Voice `Id задачи`;
- `03_crm_offer_rows_by_voice_id_prpr.xlsx` — исходные CRM-строки продуктовых предложений, совпавшие с Voice `Id ПрПр`;
- `04_match_export_audit.xlsx` — логи чтения, автоопределение файлов/колонок, пересечения ключей, несмэтченные ключи.

Важно: это **не** финальный анализ CRM-истории. Здесь нет `ucpid`, окон дат и интерпретации статусов.
Это только проверяемый raw-слой: можно открыть Excel и пройти из Voice `Id задачи` / `Id ПрПр`
в исходную строку CRM.

In [ ]:
from __future__ import annotations

from pathlib import Path
import ast
import csv
import re
import warnings
from collections import defaultdict
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 260)

## 1. Настройки

In [ ]:
BASE_DIR = Path.cwd()

# Если автоопределение не сработает, укажите явно, например:
# VOICE_MATCH_XLSX = BASE_DIR / "Для метчинга.xlsx"
VOICE_MATCH_XLSX = None

# Если нужно ограничить список CRM-файлов, укажите явно:
# CRM_CSV_FILES = [BASE_DIR / "part-....csv", BASE_DIR / "part-....csv"]
CRM_CSV_FILES = None

OUT_DIR = BASE_DIR / "omega_raw_match_export"
OUT_DIR.mkdir(exist_ok=True, parents=True)

print("Рабочая папка:", BASE_DIR)
print("Папка результата:", OUT_DIR)

## 2. Утилиты

In [ ]:
NULL_LIKE = {"", "NULL", "[NULL]", "None", "nan", "NaN", "NaT", "[]"}


def clean(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


def canon_name(name: Any) -> str:
    s = clean(name).lower().replace("ё", "е")
    s = re.sub(r"[\s_\-./()]+", "", s)
    return s


def find_col(
    df: pd.DataFrame,
    exact: list[str] | None = None,
    regex: list[str] | None = None,
    required: bool = False,
    label: str = "",
) -> str | None:
    exact = exact or []
    regex = regex or []
    exact_canons = {canon_name(x) for x in exact}
    for col in df.columns:
        if canon_name(col) in exact_canons:
            return col
    for pattern in regex:
        rx = re.compile(pattern, flags=re.I)
        for col in df.columns:
            if rx.search(canon_name(col)) or rx.search(clean(col)):
                return col
    if required:
        raise KeyError(f"Не найдена обязательная колонка {label or exact or regex}. Доступные: {list(df.columns)}")
    return None


def strip_wrappers(value: Any) -> str:
    text = clean(value)
    for _ in range(4):
        old = text
        text = text.strip().strip("'").strip('"').strip()
        if text.startswith("[") and text.endswith("]"):
            text = text[1:-1].strip()
        if text.startswith("(") and text.endswith(")"):
            text = text[1:-1].strip()
        if text == old:
            break
    return text


def explode_cell(value: Any) -> list[str]:
    text = clean(value)
    if not text or text in NULL_LIKE:
        return []
    if (text.startswith("[") and text.endswith("]")) or (text.startswith("(") and text.endswith(")")):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set)):
                out: list[str] = []
                for item in parsed:
                    out.extend(explode_cell(item))
                return out
            return explode_cell(parsed)
        except Exception:
            pass
    text = strip_wrappers(text)
    if not text or text in NULL_LIKE:
        return []
    # Иногда в выгрузке встречается строка вида "A, A"; UUID/числа запятых не содержат.
    if "," in text:
        parts = [strip_wrappers(x) for x in text.split(",")]
        return [x for x in parts if x and x not in NULL_LIKE]
    return [text]


def norm_key(value: Any) -> str:
    text = strip_wrappers(value)
    if not text or text in NULL_LIKE:
        return ""
    return re.sub(r"\s+", "", text).upper()


def key_variants(value: Any) -> list[str]:
    variants: list[str] = []
    for part in explode_cell(value):
        k = norm_key(part)
        if k:
            variants.append(k)
    # сохраняем порядок, убираем дубли
    return list(dict.fromkeys(variants))


def pct(num: int | float, den: int | float) -> float:
    return round(float(num) / float(den) * 100, 1) if den else 0.0


def trunc(value: Any, limit: int = 120) -> str:
    s = re.sub(r"\s+", " ", clean(value)).strip()
    return s if len(s) <= limit else s[: limit - 1].rstrip() + "…"


def write_xlsx(path: Path, sheets: dict[str, pd.DataFrame]) -> None:
    path.parent.mkdir(exist_ok=True, parents=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet, df in sheets.items():
            safe_sheet = sheet[:31]
            df.to_excel(writer, sheet_name=safe_sheet, index=False)
            ws = writer.book[safe_sheet]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for col_cells in ws.columns:
                header = str(col_cells[0].value or "")
                width = min(max(len(header) + 2, 12), 60)
                ws.column_dimensions[col_cells[0].column_letter].width = width
    print("saved:", path)

## 3. Читаем Voice-ключи для матчинга

In [ ]:
def find_voice_match_file() -> Path:
    if VOICE_MATCH_XLSX is not None:
        return Path(VOICE_MATCH_XLSX)
    candidates = []
    for path in BASE_DIR.glob("*.xlsx"):
        name = path.name.lower()
        if path.name.startswith("~$"):
            continue
        if "метч" in name or "match" in name or path.name == "Для метчинга.xlsx":
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError("Не найден Excel с ключами Voice. Положите рядом 'Для метчинга.xlsx' или задайте VOICE_MATCH_XLSX.")
    # Предпочитаем точное имя, иначе самый свежий файл.
    exact = [p for p in candidates if p.name == "Для метчинга.xlsx"]
    return exact[0] if exact else max(candidates, key=lambda p: p.stat().st_mtime)


voice_path = find_voice_match_file()
voice = pd.read_excel(voice_path, dtype=str)
print("Voice match file:", voice_path.name, voice.shape)
display(voice.head())

voice_cols = {
    "ucid": find_col(voice, exact=["ucid"], required=True, label="Voice ucid"),
    "call_id": find_col(voice, exact=["Id звонка", "ID звонка", "id звонка", "call_id"]),
    "activity_date": find_col(voice, exact=["Дата активности", "Дата звонка", "date"]),
    "task_id": find_col(voice, exact=["Id задачи", "ID задачи", "id задачи"], regex=[r"(id|ид).*задач", r"task.*id"], required=True, label="Voice Id задачи"),
    "offer_id": find_col(voice, exact=["Id ПрПр", "ID ПрПр", "id прпр"], regex=[r"(id|ид).*прпр", r"product.*offer"], required=True, label="Voice Id ПрПр"),
    "org_id": find_col(voice, exact=["Id Организации (стп/ЕКП)", "Id Организации (crm/ЕКП)", "Id Организации", "Id организации"], regex=[r"(id|ид).*орган", r"ucp", r"екп", r"стп"]),
}
print("Detected Voice cols:", voice_cols)

voice_small = pd.DataFrame({
    "voice_row_index": voice.index,
    "ucid": voice[voice_cols["ucid"]].map(clean),
    "Id звонка": voice[voice_cols["call_id"]].map(clean) if voice_cols["call_id"] else "",
    "Дата активности": voice[voice_cols["activity_date"]].map(clean) if voice_cols["activity_date"] else "",
    "Id задачи": voice[voice_cols["task_id"]].map(clean),
    "Id ПрПр": voice[voice_cols["offer_id"]].map(clean),
    "Id Организации": voice[voice_cols["org_id"]].map(clean) if voice_cols["org_id"] else "",
})


def build_voice_key_map(df: pd.DataFrame, source_col: str, value_col: str) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        for key in key_variants(row[value_col]):
            rows.append({
                "match_key": key,
                "voice_row_index": row["voice_row_index"],
                "ucid": row["ucid"],
                "Id звонка": row["Id звонка"],
                "Дата активности": row["Дата активности"],
                "voice_source_col": source_col,
                "voice_source_value": row[value_col],
            })
    return pd.DataFrame(rows).drop_duplicates()


voice_task_keys = build_voice_key_map(voice_small, "Id задачи", "Id задачи")
voice_offer_keys = build_voice_key_map(voice_small, "Id ПрПр", "Id ПрПр")

print("Voice rows:", len(voice_small))
print("Voice task keys:", voice_task_keys["match_key"].nunique() if len(voice_task_keys) else 0)
print("Voice offer keys:", voice_offer_keys["match_key"].nunique() if len(voice_offer_keys) else 0)

display(voice_small.head())
display(voice_task_keys.head())
display(voice_offer_keys.head())

## 4. Читаем CRM part-*.csv

In [ ]:
def read_crm_csv(path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Повторяем устойчивое чтение, которым уже читали файлы Ани: tab + cp1251 + QUOTE_NONE.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", pd.errors.ParserWarning)
        df = pd.read_csv(
            path,
            sep="\t",
            encoding="cp1251",
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="warn",
            dtype=str,
        )
    parser_messages = [str(w.message) for w in caught if issubclass(w.category, pd.errors.ParserWarning)]
    info = {
        "file": path.name,
        "rows_read": len(df),
        "columns": len(df.columns),
        "parser_warnings": len(parser_messages),
        "first_warning": parser_messages[0] if parser_messages else "",
    }
    return df, info


crm_paths = [Path(p) for p in CRM_CSV_FILES] if CRM_CSV_FILES is not None else sorted(BASE_DIR.glob("part-*.csv"))
if not crm_paths:
    raise FileNotFoundError("Не найдены CRM-файлы part-*.csv рядом с ноутбуком.")

crm_frames: dict[str, pd.DataFrame] = {}
read_log = []
for path in crm_paths:
    df, info = read_crm_csv(path)
    crm_frames[path.name] = df
    read_log.append(info)
    print(path.name, df.shape)

crm_read_log = pd.DataFrame(read_log)
display(crm_read_log)

## 5. Определяем файл задач и файл продуктовых предложений

In [ ]:
def profile_crm_file(name: str, df: pd.DataFrame) -> dict[str, Any]:
    taskid = find_col(df, exact=["taskid", "taskId"], regex=[r"^taskid$", r"task.*id"])
    product_offer_id = find_col(df, exact=["productOfferId", "productofferid"], regex=[r"product.*offer.*id"])
    key = find_col(df, exact=["key"], regex=[r"^key$"])
    ucpid = find_col(df, exact=["ucpid", "ucpId"], regex=[r"^ucpid$"])
    return {
        "file": name,
        "rows": len(df),
        "columns": len(df.columns),
        "taskid_col": taskid or "",
        "productOfferId_col": product_offer_id or "",
        "key_col": key or "",
        "ucpid_col": ucpid or "",
        "detected_entity": "task" if taskid else ("offer" if product_offer_id else "unknown"),
    }


crm_profiles = pd.DataFrame([profile_crm_file(name, df) for name, df in crm_frames.items()])
display(crm_profiles)

task_files = crm_profiles[crm_profiles["taskid_col"].ne("")]["file"].tolist()
offer_files = crm_profiles[crm_profiles["productOfferId_col"].ne("")]["file"].tolist()

if not task_files:
    raise ValueError("Не найден CRM-файл задач: нет колонки taskid.")
if not offer_files:
    raise ValueError("Не найден CRM-файл продуктовых предложений: нет колонки productOfferId.")

print("Task files:", task_files)
print("Offer files:", offer_files)

## 6. Фильтруем CRM-строки по Voice Id задачи / Id ПрПр

In [ ]:
def build_voice_lookup(voice_key_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    return {key: part.copy() for key, part in voice_key_df.groupby("match_key", dropna=False)}


def filter_crm_by_voice_keys(
    entity: str,
    files: list[str],
    crm_profiles: pd.DataFrame,
    voice_key_df: pd.DataFrame,
    primary_col_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    voice_lookup = build_voice_lookup(voice_key_df)
    voice_keys = set(voice_lookup)
    raw_parts = []
    helper_rows = []
    overlap_rows = []
    matched_voice_keys: set[str] = set()

    for file in files:
        df = crm_frames[file]
        profile = crm_profiles[crm_profiles["file"].eq(file)].iloc[0].to_dict()
        crm_id_cols = []
        primary_col = profile.get(primary_col_name) or ""
        key_col = profile.get("key_col") or ""
        if primary_col:
            crm_id_cols.append(primary_col)
        if key_col and key_col not in crm_id_cols:
            crm_id_cols.append(key_col)

        file_keys: set[str] = set()
        matched_idx: set[int] = set()

        for idx, row in df.iterrows():
            row_matches = []
            for col in crm_id_cols:
                for key in key_variants(row.get(col)):
                    if key:
                        file_keys.add(key)
                    if key in voice_keys:
                        matched_idx.add(idx)
                        matched_voice_keys.add(key)
                        vpart = voice_lookup[key]
                        row_matches.append((col, key, vpart))
            for col, key, vpart in row_matches:
                helper_rows.append({
                    "entity": entity,
                    "crm_file": file,
                    "crm_row_index": idx,
                    "crm_match_column": col,
                    "crm_match_key": key,
                    "matched_voice_rows": len(vpart),
                    "matched_voice_ucids": "; ".join(vpart["ucid"].dropna().astype(str).drop_duplicates().head(20)),
                    "matched_voice_call_ids": "; ".join(vpart["Id звонка"].dropna().astype(str).drop_duplicates().head(20)),
                    "matched_voice_source_values": "; ".join(vpart["voice_source_value"].dropna().astype(str).drop_duplicates().head(20)),
                })

        matched = df.loc[sorted(matched_idx)].copy() if matched_idx else df.iloc[0:0].copy()
        if len(matched):
            raw_parts.append(matched)
        overlap_rows.append({
            "entity": entity,
            "crm_file": file,
            "crm_id_columns_checked": ", ".join(crm_id_cols),
            "voice_keys": len(voice_keys),
            "crm_keys": len(file_keys),
            "intersection_keys": len(file_keys & voice_keys),
            "matched_crm_rows": len(matched),
            "matched_voice_keys_so_far": len(matched_voice_keys),
            "example_intersection_keys": "; ".join(sorted(file_keys & voice_keys)[:10]),
            "example_voice_keys_without_this_crm_file": "; ".join(sorted(voice_keys - file_keys)[:10]),
        })

    raw_out = pd.concat(raw_parts, ignore_index=False).reset_index(drop=True) if raw_parts else pd.DataFrame()
    helper = pd.DataFrame(helper_rows)
    overlap = pd.DataFrame(overlap_rows)
    unmatched_voice = voice_key_df[~voice_key_df["match_key"].isin(matched_voice_keys)].copy()
    return raw_out, helper, overlap, unmatched_voice


task_raw, task_helper, task_overlap, task_unmatched_voice = filter_crm_by_voice_keys(
    entity="task",
    files=task_files,
    crm_profiles=crm_profiles,
    voice_key_df=voice_task_keys,
    primary_col_name="taskid_col",
)

offer_raw, offer_helper, offer_overlap, offer_unmatched_voice = filter_crm_by_voice_keys(
    entity="offer",
    files=offer_files,
    crm_profiles=crm_profiles,
    voice_key_df=voice_offer_keys,
    primary_col_name="productOfferId_col",
)

print("Matched task CRM rows:", len(task_raw))
print("Matched offer CRM rows:", len(offer_raw))
display(task_overlap)
display(offer_overlap)

## 7. Сохраняем Excel-файлы для ручной сверки

In [ ]:
summary = pd.DataFrame([
    {"Показатель": "Voice rows", "N": len(voice_small)},
    {"Показатель": "Voice unique Id задачи keys", "N": voice_task_keys["match_key"].nunique() if len(voice_task_keys) else 0},
    {"Показатель": "Voice unique Id ПрПр keys", "N": voice_offer_keys["match_key"].nunique() if len(voice_offer_keys) else 0},
    {"Показатель": "Matched raw task CRM rows", "N": len(task_raw)},
    {"Показатель": "Matched raw offer CRM rows", "N": len(offer_raw)},
    {"Показатель": "Unmatched Voice Id задачи keys", "N": task_unmatched_voice["match_key"].nunique() if len(task_unmatched_voice) else 0},
    {"Показатель": "Unmatched Voice Id ПрПр keys", "N": offer_unmatched_voice["match_key"].nunique() if len(offer_unmatched_voice) else 0},
])

write_xlsx(
    OUT_DIR / "01_voice_match_keys.xlsx",
    {
        "Voice rows": voice_small,
        "Voice Id задачи keys": voice_task_keys,
        "Voice Id ПрПр keys": voice_offer_keys,
    },
)

write_xlsx(
    OUT_DIR / "02_crm_task_rows_by_voice_id_task.xlsx",
    {
        "CRM task rows original": task_raw,
        "Match helper": task_helper,
        "Unmatched Voice task keys": task_unmatched_voice.head(5000),
    },
)

write_xlsx(
    OUT_DIR / "03_crm_offer_rows_by_voice_id_prpr.xlsx",
    {
        "CRM offer rows original": offer_raw,
        "Match helper": offer_helper,
        "Unmatched Voice offer keys": offer_unmatched_voice.head(5000),
    },
)

write_xlsx(
    OUT_DIR / "04_match_export_audit.xlsx",
    {
        "Summary": summary,
        "CRM read log": crm_read_log,
        "CRM file profiles": crm_profiles,
        "Task key overlap": task_overlap,
        "Offer key overlap": offer_overlap,
        "Task helper sample": task_helper.head(1000),
        "Offer helper sample": offer_helper.head(1000),
    },
)

display(summary)
print("Готово. Файлы сохранены в:", OUT_DIR)

## Как глазами проверять

1. Откройте `01_voice_match_keys.xlsx` и возьмите конкретный `Id задачи` или `Id ПрПр` из Voice.
2. Откройте:
   - `02_crm_task_rows_by_voice_id_task.xlsx`, если проверяете `Id задачи`;
   - `03_crm_offer_rows_by_voice_id_prpr.xlsx`, если проверяете `Id ПрПр`.
3. На листе `CRM ... rows original` ищите исходную CRM-строку.
4. На листе `Match helper` видно, через какую CRM-колонку она совпала и с какими `ucid` / `Id звонка`.

Лист `CRM ... rows original` сохраняет исходные CRM-колонки без добавления наших аналитических полей.